# Customer Dimention table with SCD-2 (Slowly changing dimentions- Type 2) implementation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_customers = spark.table("silver.customers")
silver_orders = spark.table("silver.orders")
silver_customers.limit(2).display()

In [0]:
silver_customers.select(F.countDistinct("customer_unique_id")).show()

### Remove lineage columns _load_timestamp and _source_file

In [0]:
silver_customers = silver_customers.drop("_load_timestamp").drop("_source_file") 

In [0]:
silver_customers.printSchema()

In [0]:
cust = silver_customers.join(
    silver_orders.select("customer_id", "order_purchase_timestamp"),
    on="customer_id", how="left"
)

# one row per PERSON: keep their most recent order's attributes
w = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc())
seed = (cust
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select(
        "customer_unique_id",
        "customer_zip_code_prefix", "customer_city", "customer_state"
    )
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
)

In [0]:
seed.limit(5).display()

### Write to gold.dim_customer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold

In [0]:
#  %sql
# DROP TABLE IF EXISTS gold.dim_customer


In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_customer (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_unique_id STRING,
  customer_zip_code_prefix STRING,
  customer_city STRING,
  customer_state STRING,
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN
)

In [0]:
%sql
DESCRIBE gold.dim_customer

In [0]:
seed.write.format("delta").mode("overwrite").option("inferSchema", True).saveAsTable("gold.dim_customer")

In [0]:
spark.table("gold.dim_customer").limit(5).display()

In [0]:
%sql
-- 2. Of those, how many are marked current per person? This is the real test.
SELECT customer_unique_id, COUNT(*) AS current_versions
FROM gold.dim_customer
WHERE is_current = true
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY current_versions DESC
LIMIT 10;

## Phase2: SCD2 Merge-Into

In [0]:
df = spark.table("silver.customers").limit(5)
df.display()

In [0]:
df_new = df.withColumn("rn", F.row_number().over(Window.orderBy(F.lit(1))))
df_new.display()

In [0]:
lookup = spark.createDataFrame(
    [
        (1, "01310", "Sao Paulo", "SP"),
        (2, "20040", "Rio de Janeiro", "RJ"),
        (3, "30130", "Belo Horizonte", "MG"),
        (4, "80010", "Curitiba", "PR"),
        (5, "90010", "Porto Alegre", "RS")
    ],
    ["rn", "customer_zip_code_prefix", "customer_city", "customer_state"]
)

lookup.display()

In [0]:
df_new = df_new.drop(
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)
df_new = df_new.join(lookup, on="rn", how="left")
df_new.display()

In [0]:
df_new = df_new.drop("_load_timestamp").drop("_source_file").drop("rn").drop("customer_id")

In [0]:
df_new.display()

## MERGE-INSERT

In [0]:

# --- Stage the two-copy source ---
# Copy A: the "expire" rows — these will MATCH current target rows.
expire_copy = df_new.withColumn("mergeKey", F.col("customer_unique_id"))

# Copy B: the "insert" rows — mergeKey = null so they NEVER match -> insert.
# We only need this copy for customers who actually changed, but unioning
# all is fine; unchanged ones won't survive the MATCHED condition anyway.
insert_copy = df_new.withColumn("mergeKey", F.lit(None).cast("string"))

staged = expire_copy.unionByName(insert_copy)
staged.display()

In [0]:
staged.createOrReplaceTempView("staged_customers")

## THIS IS THE MOST IMPORTANT MERGE INSERT OPERATION IN SCD 2 LAYER

In [0]:
%sql
MERGE INTO gold.dim_customer AS target
USING staged_customers AS source
ON  target.customer_unique_id = source.mergeKey
    AND target.is_current = true

-- EXPIRE: tracked attributes changed
WHEN MATCHED AND (
       target.customer_city <> source.customer_city
    OR target.customer_state <> source.customer_state
    OR target.customer_zip_code_prefix <> source.customer_zip_code_prefix
) THEN
  UPDATE SET
    target.end_date   = current_date(),
    target.is_current = false

-- INSERT: new current version
WHEN NOT MATCHED THEN
  INSERT (
      customer_unique_id,
      customer_zip_code_prefix,
      customer_city,
      customer_state,
      effective_date,
      end_date,
      is_current
  )
  VALUES (
      source.customer_unique_id,
      source.customer_zip_code_prefix,
      source.customer_city,
      source.customer_state,
      current_date(),
      DATE'9999-12-31',
      true
  );

In [0]:
spark.table("gold.dim_customer")\
    .filter((F.col("customer_unique_id") == "248ffe10d632bebe4f7267f1f44844c9") | (F.col("customer_unique_id") == "b0015e09bb4b6e47c52844fab5fb6638")).display()

In [0]:
%sql
SELECT customer_unique_id, COUNT(*) AS versions
FROM gold.dim_customer
GROUP BY customer_unique_id
HAVING COUNT(*) > 1